# HARNESS Notebook

Este notebook usa os módulos Python do projeto (sem `!pip install`).


## Pré-requisitos
1. `pip install -r requirements.txt`
2. `export GEMINI_API_KEY=...`


In [ ]:
import numpy as np

from harness.core import (
    Harness,
    MiniFaiss,
    Tokenizer,
    TurboQuant,
    evaluate_harness,
)
from harness.providers import GeminiEmbedder, GeminiLLM, configure_gemini_api_key, load_model_with_fallback


In [ ]:
configure_gemini_api_key()
model = load_model_with_fallback("models/gemma-3-27b-it")
embed_fn = GeminiEmbedder(output_dimensionality=768)

docs = [
    "TurboQuant reduz custo de embeddings",
    "MiniFaiss permite busca vetorial leve",
    "Gemini 3 tem raciocínio avançado",
    "LLMs podem ser avaliados com harness",
]


In [ ]:
retriever = MiniFaiss(dim=768)
vectors = np.array([embed_fn(d) for d in docs])

tq = TurboQuant(bits=8)
tq.fit(vectors)

for d, vec in zip(docs, vectors):
    retriever.add(tq.dequantize(tq.quantize(vec)), d)

harness = Harness(
    llm=GeminiLLM(model),
    retriever=retriever,
    tokenizer=Tokenizer(),
    embed_fn=embed_fn,
)


In [ ]:
result = harness.run("O que é TurboQuant?")
print(result["response"])
print("Tokens totais:", result["tokens"])


In [ ]:
queries = [
    ("Como reduzir custo de embeddings?", "TurboQuant"),
    ("Como fazer busca vetorial leve?", "MiniFaiss"),
]

metrics = evaluate_harness(
    harness,
    queries,
    top_k=3,
    input_cost_per_1k_tokens=0.0003,
    output_cost_per_1k_tokens=0.0006,
)
metrics
